# Exploratory Data Analysis

<div style="text-align: justify">

The following notebook is dedicated to exploratory data analysis for the <b>Tau Supersymmetry</b> search analysis. EDA is performed on the rectangularized MC DataFrame produced by the feature engineering pipeline, and covers data quality, class balance, feature correlations, and feature distributions.

</div>

## Analysis Steps

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis configuration |
| Load | `io.load_dataframe` | Read mc.parquet from feature engineering output |
| Labels | — | Derive class names from eventOrigin |
| Missing | `eda.checks.summarize_missing` | Check for remaining NaN values |
| Zeros | `eda.checks.summarize_zeros` | Check zero-padding distribution across features |
| Ranges | `eda.checks.summarize_feature_ranges` | Per-class min, max, mean and std per feature |
| Balance | `eda.plots.plot_class_balance` | Event counts before and after class weighting |
| Correlation | `eda.plots.plot_correlation_matrix` | Pearson feature correlation heatmap |
| Distributions | `eda.plots.plot_feature_distributions` | Per-class normalized feature histograms |

The same analysis is available as a CLI via `python eda.py` or `make eda`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)

Data Visualization:
* [Matplotlib](https://matplotlib.org/)
* [Seaborn](https://seaborn.pydata.org/)

Serialization:
* [Apache Parquet](https://parquet.apache.org/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, samples, features) are defined in `configs/` and can be overridden here.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")

Resolving the input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
plots_dir = path / output_paths["plots_dir"] / "eda"
plots_dir.mkdir(parents=True, exist_ok=True)

## Deserialization

Loading the MC DataFrame produced by the feature engineering pipeline.

In [ ]:
from src.processing.io import load_dataframe

df_mc = load_dataframe(dataframes_dir / "mc.parquet")

## Class Labels

Deriving ordered class names from `eventOrigin` for use in plots and tables.

In [ ]:
class_names = df_mc.groupby("class")["eventOrigin"].first().sort_index().tolist()
class_names

## Data Quality

### Missing Values

Checking for remaining NaN values after padding fill. An empty table indicates a clean DataFrame.

In [ ]:
from src.eda.checks import summarize_missing

summarize_missing(df_mc)

### Zero Values

Checking the fraction of zero values per feature. High fractions in padded columns (e.g. `jet_pt_2`) are expected — they reflect events with fewer objects than the padding threshold.

In [ ]:
from src.eda.checks import summarize_zeros

summarize_zeros(df_mc, exclude=["class", "class_weight", "tau_n"])

### Feature Ranges

Inspecting per-class min, max, mean and std for all numeric features. Useful for catching outliers or unexpected value ranges.

In [ ]:
from src.eda.checks import summarize_feature_ranges

summarize_feature_ranges(df_mc)

## Class Balance

Plotting raw and class-weighted event counts per class. The weighted panel should show equal bars, confirming that `class_weight` correctly compensates for class imbalance.

In [ ]:
from src.eda.plots import plot_class_balance
from src.visualization.plots import save_figure

fig = plot_class_balance(df_mc, class_names=class_names)
save_figure(fig, plots_dir / "class_balance.png")
fig.show()

## Correlations

Plotting the Pearson correlation matrix for all training features. Highly correlated feature pairs (|r| > 0.9) are candidates for removal to reduce redundancy. Annotations are shown only for matrices with 30 or fewer features.

In [ ]:
from src.eda.plots import plot_correlation_matrix
from src.visualization.plots import save_figure

fig = plot_correlation_matrix(df_mc)
save_figure(fig, plots_dir / "correlation_matrix.png")
fig.show()

## Feature Distributions

Plotting per-class normalized histograms for a selection of features. Modify `features` to inspect any subset of columns.

In [ ]:
from src.eda.plots import plot_feature_distributions
from src.visualization.plots import save_figure

training_cols = [
    c for c in df_mc.select_dtypes(include="number").columns
    if c not in {"class", "class_weight", "tau_n"}
]
features = training_cols[:12]  # adjust as needed

fig = plot_feature_distributions(df_mc, features=features, class_names=class_names)
save_figure(fig, plots_dir / "feature_distributions.png")
fig.show()